<a href="https://colab.research.google.com/github/Omarnagyafifi1/data-science-and-machine-learning-projects/blob/master/Text_classification_project_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("rmisra/news-category-dataset")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'news-category-dataset' dataset.
Path to dataset files: /kaggle/input/news-category-dataset


In [2]:
import pandas as pd
import os

# Construct the full path to the data file, assuming a common filename for this dataset
data_file_path = os.path.join(path, "News_Category_Dataset_v3.json")

# Check if the file exists before attempting to read
if os.path.exists(data_file_path):
    print(f"Reading data from: {data_file_path}")
    # Read the JSON file, assuming it's a line-delimited JSON
    df = pd.read_json(data_file_path, lines=True)
    print("Data loaded successfully. Here are the first 5 rows:")
    print(df.head())
    print("\nShape of the dataset:", df.shape)
else:
    print(f"Data file not found at {data_file_path}. Listing files in {path} for inspection:")
    for file in os.listdir(path):
        print(file)
    print("Please specify the correct filename to read, or check the contents of the dataset.")

Reading data from: /kaggle/input/news-category-dataset/News_Category_Dataset_v3.json
Data loaded successfully. Here are the first 5 rows:
                                                link  \
0  https://www.huffpost.com/entry/covid-boosters-...   
1  https://www.huffpost.com/entry/american-airlin...   
2  https://www.huffpost.com/entry/funniest-tweets...   
3  https://www.huffpost.com/entry/funniest-parent...   
4  https://www.huffpost.com/entry/amy-cooper-lose...   

                                            headline   category  \
0  Over 4 Million Americans Roll Up Sleeves For O...  U.S. NEWS   
1  American Airlines Flyer Charged, Banned For Li...  U.S. NEWS   
2  23 Of The Funniest Tweets About Cats And Dogs ...     COMEDY   
3  The Funniest Tweets From Parents This Week (Se...  PARENTING   
4  Woman Who Called Cops On Black Bird-Watcher Lo...  U.S. NEWS   

                                   short_description               authors  \
0  Health experts said it is too early to pr

In [3]:
df=df[['headline','category']]

In [4]:
df.head()

,headline,category
0,Over 4 Million Americans Roll Up Sleeves For O...,U.S. NEWS
1,"American Airlines Flyer Charged, Banned For Li...",U.S. NEWS
2,23 Of The Funniest Tweets About Cats And Dogs ...,COMEDY
3,The Funniest Tweets From Parents This Week (Se...,PARENTING
4,Woman Who Called Cops On Black Bird-Watcher Lo...,U.S. NEWS


In [5]:
df.isna().sum()

,0
headline,0
category,0


In [6]:
df.duplicated().sum()

np.int64(1419)

In [7]:
df.category.value_counts()

,count
category,
POLITICS,35602
WELLNESS,17945
ENTERTAINMENT,17362
TRAVEL,9900
STYLE & BEAUTY,9814
PARENTING,8791
HEALTHY LIVING,6694
QUEER VOICES,6347
FOOD & DRINK,6340


In [8]:
selected_columns=['SPORTS','CRIME','COMEDY','EDUCATION']
df_new= df[df['category'].isin(selected_columns)].reset_index(drop=True)

In [9]:
df_new.headline.shape,df_new.category.shape

((15053,), (15053,))

In [10]:
df_new.category.value_counts()

,count
category,
COMEDY,5400
SPORTS,5077
CRIME,3562
EDUCATION,1014


In [11]:
#the column of category is un balanced dataset so we will use the under sampling techique
min_samples=1014
df_business=df_new[df_new.category=='COMEDY'].sample(min_samples,random_state=2022)
df_sport=df_new[df_new.category=='SPORTS'].sample(min_samples,random_state=2022)
df_crime=df_new[df_new.category=='CRIME'].sample(min_samples,random_state=2022)
df_science=df_new[df_new.category=='EDUCATION'].sample(min_samples,random_state=2022)



In [12]:
df_balanced=pd.concat([df_business,df_sport,df_crime,df_science],axis=0)

In [13]:
df_balanced.category.value_counts()

,count
category,
COMEDY,1014
SPORTS,1014
CRIME,1014
EDUCATION,1014


In [14]:
df_balanced['category_num']=df_balanced['category'].map({
'COMEDY':4,
'SPORTS':1,
'CRIME':2,
'EDUCATION':3
})

In [15]:
# applying preprocessing
import spacy
nlp=spacy.load('en_core_web_sm')
def processed_text(text):
    tokens=nlp(text)
    filterd_sent_list=[token.lemma_.lower() for token in tokens if  not (token.is_stop or  token.is_punct or  token.is_space)]
    filterd_sent=' '.join(filterd_sent_list)
    return filterd_sent

df_balanced['preprocessed_txt']=df_balanced['headline'].apply(processed_text)
# def preprocess(text):
#     # remove stop words and lemmatize the text
#     doc = nlp(text)
#     filtered_tokens = []
#     for token in doc:
#         if token.is_stop or token.is_punct:
#             continue
#         filtered_tokens.append(token.lemma_)

#     return " ".join(filtered_tokens)
# df_balanced['preprocessed_txt']=df_balanced['headline'].apply(preprocess)



In [16]:
from sklearn.naive_bayes import MultinomialNB
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import CountVectorizer

from sklearn.model_selection import train_test_split
X=df_balanced['preprocessed_txt']
y=df_balanced['category_num']
X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y,test_size=0.2,random_state=2023)
clf=Pipeline([
      ('vectorizer_bow',CountVectorizer(ngram_range=(1,1))),
      ('Multi NB',MultinomialNB()) ])
clf.fit(X_train,y_train)
y_pred=clf.predict(X_test)
print(classification_report(y_test,y_pred))
print('_______________________'*3)



              precision    recall  f1-score   support

           1       0.78      0.79      0.78       203
           2       0.89      0.92      0.91       203
           3       0.84      0.85      0.85       203
           4       0.84      0.78      0.81       203

    accuracy                           0.84       812
   macro avg       0.84      0.84      0.84       812
weighted avg       0.84      0.84      0.84       812

_____________________________________________________________________


In [17]:













# from sklearn.model_selection import train_test_split
# X=df_balanced['headline']
# y=df_balanced['category_num']
# X_train,X_test,y_train,y_test=train_test_split(X,y,stratify=y,test_size=0.2,random_state=2023)



# from sklearn.naive_bayes import MultinomialNB
# from sklearn.ensemble import RandomForestClassifier
# from sklearn.metrics import classification_report
# from sklearn.pipeline import Pipeline
# from sklearn.feature_extraction.text import CountVectorizer

# clf=Pipeline([
# ('vectorizer_bow',CountVectorizer(ngram_range=(1,1))),
# ('Multi NB',MultinomialNB()) ])
# clf.fit(X_train,y_train)
# y_pred=clf.predict(X_test)
# print(classification_report(y_test,y_pred))


In [18]:
import joblib
joblib.dump(clf,'Text_classification_model')

['Text_classification_model']

In [19]:
import gradio as gr
import joblib

# Load the trained model
loaded_model = joblib.load('Text_classification_model')

# Define the category mapping (reverse of what was used for training)
category_map = {
    4: 'COMEDY',
    1: 'SPORTS',
    2: 'CRIME',
    3: 'EDUCATION'
}

def predict_category(headline):
    # The loaded_model is a pipeline that includes CountVectorizer and MultinomialNB
    # It expects a list of strings for prediction
    prediction_numeric = loaded_model.predict([headline])[0]
    predicted_category = category_map[prediction_numeric]
    return predicted_category

# Create the Gradio interface
iface = gr.Interface(
    fn=predict_category,
    inputs=gr.Textbox(lines=2, placeholder="Enter a news headline here..."),
    outputs="text",
    title="News Headline Classifier",
    description="Enter a news headline to classify it into one of four categories: COMEDY, SPORTS, CRIME, or EDUCATION."
)

# Launch the app
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://213c733516e4b61e46.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
